In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [2]:
# ─── Install Dependencies ───
!pip install -q xgboost lightgbm catboost
print("Environment ready.")

Environment ready.


In [3]:
# ─── Imports ───
import warnings, time, gc
warnings.filterwarnings("ignore")

# --- Import Data Analysis & Visualization Libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# --- Import Data Splitting, Cross-Validation & Feature Scaling ---
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# --- Import Machine Learning Classification Models ---
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# --- Import Classification Evaluation Metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)


# ─── Plot Configuration ───
sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 11})
SEED = 42
np.random.seed(SEED)

print("✅ All libraries loaded!")
print("Environment ready.")

✅ All libraries loaded!
Environment ready.


In [5]:
# Load train.parquet
file_path = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"
df = pd.read_parquet(file_path)

print("✅ Dataset loaded!")
print("Shape:", df.shape)

# Convert to CSV and save in Kaggle working directory
csv_path = "/kaggle/working/train.csv"

df.to_csv(
    csv_path,
    index=False
)

print("✅ CSV file created successfully!")
print("Saved at:", csv_path)

# Create a clickable download link
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/train.csv"))


✅ Dataset loaded!
Shape: (2539608, 18)
✅ CSV file created successfully!
Saved at: /kaggle/working/train.csv


/kaggle/working/train.csv

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2539608 entries, 0 to 2539607
Data columns (total 18 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   ingest_lib                   object 
 1   normalized_smiles            object 
 2   inchikey                     object 
 3   inchikey14                   object 
 4   molecular_formula            object 
 5   ionization_mode              object 
 6   instrument_type              object 
 7   adduct                       object 
 8   adduct_orig                  object 
 9   precursor_mz                 float64
 10  precursor_error_ppm          float64
 11  ms2_mzs                      object 
 12  ms2_normalized_intensities   object 
 13  num_peaks                    int64  
 14  base_peak_intensity          float64
 15  collision_energy_ev          object 
 16  collision_energy_orig        object 
 17  collision_energy_orig_units  object 
dtypes: float64(3), int64(1), object(14)
memory

In [7]:
df.describe()

,precursor_mz,precursor_error_ppm,num_peaks,base_peak_intensity
count,2.539608e+06,2.535322e+06,2.539608e+06,1.376470e+06
mean,4.197696e+02,1.845726e+03,1.582158e+02,5.392101e+06
std,1.535987e+03,5.117525e+05,5.168801e+02,5.295446e+07
min,2.000000e+00,-2.106149e+01,1.000000e+00,1.000000e+00
25%,3.081410e+02,7.755261e-01,1.400000e+01,1.920400e+04
50%,3.501088e+02,1.512113e+00,4.200000e+01,1.756928e+05
75%,4.892142e+02,2.345292e+00,1.640000e+02,1.250839e+06
max,2.430333e+06,4.697715e+08,7.331800e+04,8.925277e+09


In [8]:
df.head(5)

,ingest_lib,normalized_smiles,inchikey,inchikey14,molecular_formula,ionization_mode,instrument_type,adduct,adduct_orig,precursor_mz,precursor_error_ppm,ms2_mzs,ms2_normalized_intensities,num_peaks,base_peak_intensity,collision_energy_ev,collision_energy_orig,collision_energy_orig_units
0,drug_plus,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,AAALVYBICLMAMA,C20H15N3O2,positive,None,[M+H]+,[M+H]+,330.1237,1.671398,"[92.0495, 93.0573, 94.0607, 116.107, 123.1168,...","[0.0171037726229424, 0.328091214993177, 0.0107...",52,NaN,None,None,unknown
1,drug_plus,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG,C22H23N3O4,positive,None,[M+K]+,[M+K]+,432.1320,1.302193,"[82.1451, 89.0599, 158.9638, 248.0836, 250.098...","[0.00144055962319783, 0.959302375462881, 0.5, ...",28,NaN,None,None,unknown
2,drug_plus,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR-UHFFFAOYSA-N,AANFHDFOMFRLLR,C23H26FNO4,positive,None,[M+Na]+,[M+Na]+,422.1738,1.316437,"[91.0539, 111.117, 131.0856, 150.035, 171.0803...","[0.00262159533923045, 0.0184752871374318, 0.00...",41,NaN,None,None,unknown
3,drug_plus,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE-UHFFFAOYSA-N,AAOVKJBEBIDNHE,C16H13ClN2O,positive,None,[M+H]+,[M+H]+,285.0789,1.984579,"[58.0287, 65.0386, 77.0386, 89.0386, 90.0464, ...","[0.00908241772391234, 0.00183110584152731, 0.0...",166,NaN,None,None,unknown
4,drug_plus,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO-UHFFFAOYSA-N,ACFIXJIJDZMPPO,C21H30N7O17P3,positive,None,[M+H]+,[M+H]+,746.0984,0.708429,"[53.622, 60.372, 62.949, 74.853, 79.885, 91.29...","[0.2838, 0.11773, 0.15448, 0.11773, 0.26637, 0...",186,NaN,None,None,unknown


In [11]:
Target = "normalized_smiles"

In [14]:
# --- Encode All Categorical Columns ---

from sklearn.preprocessing import OrdinalEncoder

categorical_cols = df.select_dtypes(include="int64").columns

encoder = OrdinalEncoder()

df[categorical_cols] = encoder.fit_transform(
    df[categorical_cols]
)

print(categorical_cols)
df.head()

Index(['num_peaks'], dtype='object')


,ingest_lib,normalized_smiles,inchikey,inchikey14,molecular_formula,ionization_mode,instrument_type,adduct,adduct_orig,precursor_mz,precursor_error_ppm,ms2_mzs,ms2_normalized_intensities,num_peaks,base_peak_intensity,collision_energy_ev,collision_energy_orig,collision_energy_orig_units
0,drug_plus,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,AAALVYBICLMAMA,C20H15N3O2,positive,None,[M+H]+,[M+H]+,330.1237,1.671398,"[92.0495, 93.0573, 94.0607, 116.107, 123.1168,...","[0.0171037726229424, 0.328091214993177, 0.0107...",51.0,NaN,None,None,unknown
1,drug_plus,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG,C22H23N3O4,positive,None,[M+K]+,[M+K]+,432.1320,1.302193,"[82.1451, 89.0599, 158.9638, 248.0836, 250.098...","[0.00144055962319783, 0.959302375462881, 0.5, ...",27.0,NaN,None,None,unknown
2,drug_plus,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR-UHFFFAOYSA-N,AANFHDFOMFRLLR,C23H26FNO4,positive,None,[M+Na]+,[M+Na]+,422.1738,1.316437,"[91.0539, 111.117, 131.0856, 150.035, 171.0803...","[0.00262159533923045, 0.0184752871374318, 0.00...",40.0,NaN,None,None,unknown
3,drug_plus,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE-UHFFFAOYSA-N,AAOVKJBEBIDNHE,C16H13ClN2O,positive,None,[M+H]+,[M+H]+,285.0789,1.984579,"[58.0287, 65.0386, 77.0386, 89.0386, 90.0464, ...","[0.00908241772391234, 0.00183110584152731, 0.0...",165.0,NaN,None,None,unknown
4,drug_plus,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO-UHFFFAOYSA-N,ACFIXJIJDZMPPO,C21H30N7O17P3,positive,None,[M+H]+,[M+H]+,746.0984,0.708429,"[53.622, 60.372, 62.949, 74.853, 79.885, 91.29...","[0.2838, 0.11773, 0.15448, 0.11773, 0.26637, 0...",185.0,NaN,None,None,unknown


In [16]:
print(f"Dataset shape : {df.shape}")
#print(f"Class balance : {dict(df[Target].value_counts())}")
print(f"Missing values: {df.isnull().sum().sum()}")
#print(f"Duplicate rows  : {df.duplicated().sum()}")
print(f"Dtypes        : {dict(df.dtypes.value_counts())}")

Dataset shape : (2539608, 18)
Missing values: 1858384
Dtypes        : {dtype('O'): np.int64(14), dtype('float64'): np.int64(4)}


In [17]:
# Create a table containing missing-value information for each column
null_table = pd.DataFrame({
    'Column': df.columns,
    
    # Count the number of missing (NaN/None) values in each column
    'Null Count': df.isna().sum().values,
    
    # Calculate the percentage of missing values in each column
    'Null Percentage (%)': (
        df.isna().sum().values / len(df) * 100
    ).round(2)
})

# Keep only the columns that contain at least one null value
null_table = null_table[null_table['Null Count'] > 0]

# Sort from highest to lowest number of null values
null_table = null_table.sort_values(
    by='Null Count',
    ascending=False
).reset_index(drop=True)

# Display the final table
display(null_table)

,Column,Null Count,Null Percentage (%)
0,base_peak_intensity,1163138,45.80
1,collision_energy_ev,337443,13.29
2,collision_energy_orig,320464,12.62
3,instrument_type,33053,1.30
4,precursor_error_ppm,4286,0.17


In [1]:
# --- Handle Missing Values ---

# Numeric columns
numeric_cols = df.select_dtypes(include=['int64']).columns

# Categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill numeric missing values with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with mode
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Check remaining missing values
print("Total Missing Values:", df.isnull().sum().sum())

NameError: name 'df' is not defined